Suddenly I found not all the variables I use is real_time generating,which means I have to lag or rolling them to make sure model could working as real_time

In [38]:
import pandas as pd

In [39]:
df=pd.read_csv('../data/processed/dashboard_dataset.csv')

In [40]:
print(df.columns)

Index(['CCGT', 'INTIRL', 'NPSHYD', 'OCGT', 'OTHER', 'PS', 'WIND',
       'demand_prediction', 'NetImbalanceVolume', 'TotalAcceptedBidVolume',
       'ccgt_wind_ratio', 'imbalance_ccgt', 'ps_imbalance', 'offer_bid_spread',
       'month', 'time', 'is_spike'],
      dtype='str')


In [41]:
df['CCGT_lag_1']=df['CCGT'].shift(1)
df['INTIRL_lag_1']=df['INTIRL'].shift(1)
df['NPSHYD_lag_1']=df['NPSHYD'].shift(1)
df['OCGT_lag_1']=df['OCGT'].shift(1)
df['OTHER_lag_1']=df['OTHER'].shift(1)
df['PS_lag_1']=df['PS'].shift(1)
df['WIND_lag_1']=df['WIND'].shift(1)
df['TotalAcceptedBidVolume_lag_1']=df['TotalAcceptedBidVolume'].shift(1)
df=df.drop(columns=['CCGT', 'INTIRL', 'NPSHYD', 'OCGT', 'OTHER', 'PS', 'WIND','TotalAcceptedBidVolume'])

In [42]:
df['ccgt_wind_ratio'] = df['CCGT_lag_1'] / df['WIND_lag_1']
df['imbalance_ccgt'] = df['NetImbalanceVolume'] / df['CCGT_lag_1']
df['ps_imbalance'] = df['PS_lag_1'] * df['NetImbalanceVolume']

In [43]:
print(df.columns)
df=df.dropna()
print(df.isna().sum())

Index(['demand_prediction', 'NetImbalanceVolume', 'ccgt_wind_ratio',
       'imbalance_ccgt', 'ps_imbalance', 'offer_bid_spread', 'month', 'time',
       'is_spike', 'CCGT_lag_1', 'INTIRL_lag_1', 'NPSHYD_lag_1', 'OCGT_lag_1',
       'OTHER_lag_1', 'PS_lag_1', 'WIND_lag_1',
       'TotalAcceptedBidVolume_lag_1'],
      dtype='str')
demand_prediction               0
NetImbalanceVolume              0
ccgt_wind_ratio                 0
imbalance_ccgt                  0
ps_imbalance                    0
offer_bid_spread                0
month                           0
time                            0
is_spike                        0
CCGT_lag_1                      0
INTIRL_lag_1                    0
NPSHYD_lag_1                    0
OCGT_lag_1                      0
OTHER_lag_1                     0
PS_lag_1                        0
WIND_lag_1                      0
TotalAcceptedBidVolume_lag_1    0
dtype: int64


In [44]:
from src.features.ttsplit_time_sequence import tt_split_time

In [45]:
x_list=df.columns.tolist()
x_list.remove('time')
x_list.remove('is_spike')

print(x_list)

['demand_prediction', 'NetImbalanceVolume', 'ccgt_wind_ratio', 'imbalance_ccgt', 'ps_imbalance', 'offer_bid_spread', 'month', 'CCGT_lag_1', 'INTIRL_lag_1', 'NPSHYD_lag_1', 'OCGT_lag_1', 'OTHER_lag_1', 'PS_lag_1', 'WIND_lag_1', 'TotalAcceptedBidVolume_lag_1']


In [46]:
x_train,x_test,y_train,y_test=tt_split_time('is_spike',x_list,df,0.8)

In [47]:
from src.models.randomforest_classification import RF_class

In [48]:
params={'n_estimators':800}
selector,selected_feature=RF_class(x_train,y_train,params)

In [49]:
from src.models.evaluate_class_model import evaluate_class

In [50]:
y_pred_rf=selector.predict(x_test)
auc,cm,clr=evaluate_class(y_test,y_pred_rf)

AUC:  0.7963052331544775
Confusion Matrix: 
 [[12232   843]
 [ 1515  2903]]
Classification Report: 
               precision    recall  f1-score   support

           0     0.8898    0.9355    0.9121     13075
           1     0.7750    0.6571    0.7112      4418

    accuracy                         0.8652     17493
   macro avg     0.8324    0.7963    0.8116     17493
weighted avg     0.8608    0.8652    0.8613     17493



In [51]:
from src.models.XGBoost_and_lightboost_classification import gradient_boost

In [52]:
weight=((y_train==0).sum())/((y_train==1).sum())
params={'n_estimators':800,
        'learning_rate':0.03,
        'scale_pos_weight':weight
        }
selector,selected_feature=gradient_boost('xgb',x_train,y_train,params)

In [53]:
y_pred_xgb=selector.predict(x_test)
auc,cm,clr=evaluate_class(y_test,y_pred_xgb)

AUC:  0.8422802423944459
Confusion Matrix: 
 [[11407  1668]
 [  830  3588]]
Classification Report: 
               precision    recall  f1-score   support

           0     0.9322    0.8724    0.9013     13075
           1     0.6826    0.8121    0.7418      4418

    accuracy                         0.8572     17493
   macro avg     0.8074    0.8423    0.8215     17493
weighted avg     0.8692    0.8572    0.8610     17493



In [54]:
params={
    'n_estimators':800,
    'learning_rate':0.03,
    'class_weight':'balanced'
}
selector,selected_feature=gradient_boost('light',x_train,y_train,params)

[LightGBM] [Info] Number of positive: 18123, number of negative: 51847
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007238 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3583
[LightGBM] [Info] Number of data points in the train set: 69970, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Info] Number of positive: 18123, number of negative: 51847
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008458 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3583
[LightGBM] [Info] Number of data points in the train set: 69970, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


In [55]:
y_pred_light=selector.predict(x_test)
auc,cm,clr=evaluate_class(y_test,y_pred_light)

C:\Users\24363\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


AUC:  0.8452015524877804
Confusion Matrix: 
 [[11439  1636]
 [  815  3603]]
Classification Report: 
               precision    recall  f1-score   support

           0     0.9335    0.8749    0.9032     13075
           1     0.6877    0.8155    0.7462      4418

    accuracy                         0.8599     17493
   macro avg     0.8106    0.8452    0.8247     17493
weighted avg     0.8714    0.8599    0.8636     17493



In [58]:
from lightgbm import LGBMClassifier

In [61]:
lgbmodel = LGBMClassifier(
            objective='binary',
            subsample=0.9,
            colsample_bytree=0.6,
            random_state=88,
            n_jobs=1,
            learning_rate=0.002,
            n_estimators=5500,
        )
lgbmodel.fit(x_train,y_train)

[LightGBM] [Info] Number of positive: 18123, number of negative: 51847
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002554 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3583
[LightGBM] [Info] Number of data points in the train set: 69970, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.259011 -> initscore=-1.051115
[LightGBM] [Info] Start training from score -1.051115


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.002
,n_estimators,5500
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [76]:
y_prob_1=lgbmodel.predict_proba(x_test)[:,1]
y_pred_1=(y_prob_1>0.3).astype(int)
auc, cm, clr = evaluate_class(y_test, y_pred_1)

AUC:  0.8510100864964898
Confusion Matrix: 
 [[11437  1638]
 [  763  3655]]
Classification Report: 
               precision    recall  f1-score   support

           0     0.9375    0.8747    0.9050     13075
           1     0.6905    0.8273    0.7528      4418

    accuracy                         0.8627     17493
   macro avg     0.8140    0.8510    0.8289     17493
weighted avg     0.8751    0.8627    0.8666     17493



In [77]:
test_result = x_test.copy()

test_result['is_spike'] = y_test.values
test_result['y_prob'] = y_prob_1
test_result['y_pred'] = y_pred_1

In [78]:
test_result['time'] = df.loc[x_test.index, 'time'].values

def get_risk_level(p):
    if p >= 0.3:
        return 'HIGH'
    elif p >= 0.2:
        return 'MEDIUM'
    elif p >= 0.1:
        return 'MODERATE'
    else:
        return 'LOW'

test_result['risk_level'] = test_result['y_prob'].apply(get_risk_level)

print(test_result.head())
print(test_result.columns)

       demand_prediction  NetImbalanceVolume  ccgt_wind_ratio  imbalance_ccgt  \
69997       24626.472500          143.405940         0.284873        0.048612   
69998       23748.526250          212.591700         0.321581        0.065564   
69999       23242.723750           73.599895         0.370463        0.019919   
70000       23917.136250         -264.695500         0.408164       -0.067276   
70001       27932.983125         -567.865450         0.423814       -0.141436   

        ps_imbalance  offer_bid_spread  month  CCGT_lag_1  INTIRL_lag_1  \
69997 -185137.068540        1087.42010      1      2950.0        -330.0   
69998 -248307.105600         779.90830      1      3242.5        -330.0   
69999  -89865.471795         763.06670      1      3695.0        -332.0   
70000  309429.039500        1022.69095      1      3934.5        -332.0   
70001  484389.228850        1217.53175      1      4015.0        -331.0   

       NPSHYD_lag_1  OCGT_lag_1  OTHER_lag_1  PS_lag_1  WIND_l

In [56]:
from src.models.FN_check import fn_check

In [65]:
fn_df,tn_df=fn_check(x_test,y_test,y_pred_light,y_prob_1,threshold=0.2)

FN bound rate: 0.36319018404907977


In [66]:
print(fn_df)

       demand_prediction  NetImbalanceVolume  ccgt_wind_ratio  imbalance_ccgt  \
69997       24626.472500          143.405940         0.284873        0.048612   
70007       38233.527500           17.298605         2.053759        0.001322   
70017       30754.371875           63.201705         1.433810        0.006427   
70018       26951.358125           30.085850         1.102852        0.004105   
70059       41545.929375           53.510600         2.029183        0.003078   
...                  ...                 ...              ...             ...   
87420       25306.562500           70.473217         3.040171        0.007361   
87425       28086.038125           18.546720         2.386069        0.001769   
87441       29273.706250          295.046305         1.181557        0.027123   
87442       26574.906875          108.768899         0.850399        0.013614   
87482       32827.376250          136.383617         0.783231        0.014585   

        ps_imbalance  offer

In [67]:
print(tn_df)

       demand_prediction  NetImbalanceVolume  ccgt_wind_ratio  imbalance_ccgt  \
69997       24626.472500          143.405940         0.284873        0.048612   
70004       37856.795625          222.706400         1.351543        0.020297   
70005       38255.555000          201.275005         1.683899        0.016860   
70006       38279.120625          224.966400         1.852840        0.018103   
70007       38233.527500           17.298605         2.053759        0.001322   
...                  ...                 ...              ...             ...   
87464       31819.960625           55.640864         2.450537        0.003209   
87482       32827.376250          136.383617         0.783231        0.014585   
87483       34674.918750          724.380375         0.777612        0.073307   
87484       36818.681875          551.970402         0.925383        0.044198   
87486       33402.640625          157.382567         0.905169        0.012022   

        ps_imbalance  offer

In [68]:
from src.extract.save_dir import saved_data_dir

In [69]:
print(df.columns)

Index(['demand_prediction', 'NetImbalanceVolume', 'ccgt_wind_ratio',
       'imbalance_ccgt', 'ps_imbalance', 'offer_bid_spread', 'month', 'time',
       'is_spike', 'CCGT_lag_1', 'INTIRL_lag_1', 'NPSHYD_lag_1', 'OCGT_lag_1',
       'OTHER_lag_1', 'PS_lag_1', 'WIND_lag_1',
       'TotalAcceptedBidVolume_lag_1'],
      dtype='str')


In [70]:
saved_data_dir('dashboard_final_dateset.csv',df,'../data/processed/')

dashboard_final_dateset.csv saved to C:\Users\24363\Desktop\powering_market_forecasting_analytics\data\processed\dashboard_final_dateset.csv


In [79]:
saved_data_dir('dashboard_test_dataset.csv',test_result,'../data/processed/')

dashboard_test_dataset.csv saved to C:\Users\24363\Desktop\powering_market_forecasting_analytics\data\processed\dashboard_test_dataset.csv
